# End to end workflow

This notebook calls the functions in `src/` in the same order as the command line pipeline: download, build features, EDA, train, simulate, analyse, latest signals.

Set `SOURCE` to `download` for real NSE data, `file` to reuse an existing `features.parquet`, or `synthetic` for a pipeline demo (use a separate `ALGO_DATA_DIR` for synthetic runs and never read its numbers as evidence).

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, Markdown, display

from src import config, features, models, pipeline, simulation

In [2]:
SOURCE = os.environ.get("NOTEBOOK_SOURCE", "download")
FAST = os.environ.get("NOTEBOOK_FAST", "0") == "1"
print(SOURCE, FAST, config.DATA_DIR)

file True C:\Users\saran\Downloads\nifty500-algo-trading\data


## 1. Download raw NSE files

In [3]:
if SOURCE == "download":
    pipeline.step_download()

## 2. Combine into one table and build features

In [4]:
frame = pipeline.step_build(SOURCE)
frame[["date", "symbol", "sector", "price", "target_reg", "target_cls"]].tail()

Feature table rows 1,236,778, columns 69, last date 2026-11-20


,date,symbol,sector,price,target_reg,target_cls
1236773,2026-11-20,ZENSARTECH,Information Technology,529.049988,NaN,NaN
1236774,2026-11-20,ZENTEC,Capital Goods,1680.900024,NaN,NaN
1236775,2026-11-20,ZETWERK,Unknown,170.630005,NaN,NaN
1236776,2026-11-20,ZFCVINDIA,Automobile and Auto Components,2191.800049,NaN,NaN
1236777,2026-11-20,ZYDUSLIFE,Healthcare,1200.000000,NaN,NaN


## 3. Exploratory analysis

In [5]:
eda_result = pipeline.step_eda()
eda_result["correlations"].head(15)

EDA written to C:\Users\saran\Downloads\nifty500-algo-trading\reports


,feature,spearman_target_reg,spearman_target_cls,abs_corr
0,vol_63,-0.049236,0.050704,0.049236
1,atr_pct_14,-0.043430,0.062650,0.043430
2,vol_21,-0.043271,0.052281,0.043271
3,intraday_range,-0.042835,0.051084,0.042835
4,vol_5,-0.040623,0.042872,0.040623
5,cs_rank_vol_21,-0.039370,0.041917,0.039370
6,log_amihud_21,-0.037428,0.021222,0.037428
7,max_ret_21,-0.037018,0.042142,0.037018
8,cs_rank_deliv_per,0.035018,0.000113,0.035018
9,deliv_per,0.034697,0.001088,0.034697


## 4. Train and tune models

In [6]:
training = pipeline.step_train(fast=FAST)
training["thresholds"]

Training on 1,236,778 rows and 137 features, fast=True


split prediction   rows  base_rate      auc  average_precision  precision_top_1pct
valid    pred_dt 165530   0.260811 0.518134           0.273614            0.303323
valid    pred_rf 165530   0.260811 0.555086           0.298272            0.367372
valid   pred_xgb 165530   0.260811 0.547537           0.292994            0.370393
valid   pred_ens 165530   0.260811 0.554385           0.297867            0.366767
 test    pred_dt 471437   0.274298 0.529577           0.294496            0.308019
 test    pred_rf 471437   0.274298 0.553771           0.313263            0.390751
 test   pred_xgb 471437   0.274298 0.550387           0.311643            0.404752
 test   pred_ens 471437   0.274298 0.554355           0.314096            0.403055


{'pred_dt': {'threshold': 0.3264112174510956,
  'quantile': 0.9,
  'valid_rows_selected': 18366,
  'valid_mean_net_return': 0.00278448686003685,
  'tstat': 5.205734866649344},
 'pred_rf': {'threshold': 0.3017481744289398,
  'quantile': 0.9,
  'valid_rows_selected': 16553,
  'valid_mean_net_return': 0.0064231413416564465,
  'tstat': 10.65211038044888},
 'pred_xgb': {'threshold': 0.30222001671791077,
  'quantile': 0.9,
  'valid_rows_selected': 16553,
  'valid_mean_net_return': 0.0033936600666493177,
  'tstat': 5.890693673283615},
 'pred_ens': {'threshold': 0.3004755139350891,
  'quantile': 0.9,
  'valid_rows_selected': 16553,
  'valid_mean_net_return': 0.005109568126499653,
  'tstat': 8.462615164022793}}

## 5. Simulate strategies and compare with benchmarks

In [7]:
table, vector = pipeline.step_simulate()
table

                               key    cagr  sharpe  max_drawdown  trades  win_rate
        pred_dt__sim1_equal_weight -0.0298 -0.3489       -0.4149  1479.0    0.4638
       pred_dt__sim2_stop_weighted -0.2041 -1.3176       -0.6215  1398.0    0.4435
 pred_dt__sim3_regime_concentrated -0.0896 -0.7381       -0.4028   597.0    0.4539
        pred_rf__sim1_equal_weight -0.0208 -0.1145       -0.4258  1479.0    0.4936
       pred_rf__sim2_stop_weighted -0.0593 -0.2566       -0.5812  1486.0    0.4711
 pred_rf__sim3_regime_concentrated  0.0596  0.1173       -0.4678   622.0    0.4759
       pred_xgb__sim1_equal_weight -0.1291 -0.4743       -0.5018  1479.0    0.4503
      pred_xgb__sim2_stop_weighted -0.2295 -0.8746       -0.6478  1513.0    0.4263
pred_xgb__sim3_regime_concentrated -0.1491 -0.5721       -0.5568   610.0    0.4475
       pred_ens__sim1_equal_weight -0.0833 -0.3141       -0.5413  1479.0    0.4767
      pred_ens__sim2_stop_weighted -0.0676 -0.2739       -0.5610  1506.0    0.4602
pred

Best strategy by CAGR: pred_rf__sim3_regime_concentrated


,key,prediction,strategy,total_return,cagr,volatility,sharpe,sortino,max_drawdown,calmar,years,trades,win_rate,avg_trade_return,avg_holding_days,stop_exits,avg_exposure
0,pred_dt__sim1_equal_weight,pred_dt,sim1_equal_weight,-0.083499,-0.029763,0.208507,-0.348933,-0.448265,-0.414891,-0.071738,2.885695,1479.0,0.463827,-0.000053,5.000000,0.0,0.791458
1,pred_dt__sim2_stop_weighted,pred_dt,sim2_stop_weighted,-0.482594,-0.204148,0.203342,-1.317641,-1.762217,-0.621486,-0.328485,2.885695,1398.0,0.443491,-0.003801,4.741774,232.0,0.776763
2,pred_dt__sim3_regime_concentrated,pred_dt,sim3_regime_concentrated,-0.237371,-0.089631,0.188555,-0.738125,-0.929167,-0.402813,-0.222513,2.885695,597.0,0.453936,-0.000916,4.787270,75.0,0.609811
3,pred_rf__sim1_equal_weight,pred_rf,sim1_equal_weight,-0.058720,-0.020752,0.314505,-0.114459,-0.160944,-0.425821,-0.048735,2.885695,1479.0,0.493577,0.000653,5.000000,0.0,0.790419
4,pred_rf__sim2_stop_weighted,pred_rf,sim2_stop_weighted,-0.161674,-0.059281,0.304768,-0.256594,-0.352893,-0.581177,-0.102002,2.885695,1486.0,0.471063,-0.000313,4.416555,428.0,0.749144
5,pred_rf__sim3_regime_concentrated,pred_rf,sim3_regime_concentrated,0.181974,0.059647,0.289657,0.117319,0.147307,-0.467835,0.127496,2.885695,622.0,0.475884,0.001749,4.427653,175.0,0.586805
6,pred_xgb__sim1_equal_weight,pred_xgb,sim1_equal_weight,-0.329036,-0.129147,0.316687,-0.474299,-0.656835,-0.501765,-0.257385,2.885695,1479.0,0.450304,-0.001468,5.000000,0.0,0.789471
7,pred_xgb__sim2_stop_weighted,pred_xgb,sim2_stop_weighted,-0.528659,-0.229454,0.310765,-0.874618,-1.216178,-0.647818,-0.354196,2.885695,1513.0,0.426305,-0.004368,4.385327,460.0,0.750870
8,pred_xgb__sim3_regime_concentrated,pred_xgb,sim3_regime_concentrated,-0.372516,-0.149133,0.306872,-0.572149,-0.663807,-0.556784,-0.267847,2.885695,610.0,0.447541,-0.002809,4.396721,174.0,0.579911
9,pred_ens__sim1_equal_weight,pred_ens,sim1_equal_weight,-0.221952,-0.083295,0.317389,-0.314139,-0.434842,-0.541286,-0.153883,2.885695,1479.0,0.476673,-0.000469,5.000000,0.0,0.789959


## 6. Error analysis

In [8]:
tables, gaps = pipeline.step_analyze()

# Findings for pred_ens at threshold 0.3005

Signals on the test period: 85892. Precision 0.330. Average forward return 0.0088.
By year: best lift 1.23 at 2025, worst lift 1.17 at 2026.
By regime: best lift 1.25 at healthy, worst lift 1.08 at weak.
By volatility_bucket: best lift 1.17 at q1_low, worst lift 1.12 at q4_high.
By liquidity_bucket: best lift 1.25 at q1_low, worst lift 1.16 at q4_high.
By sector (50 or more signals): best lift 1.42 at Construction, worst lift 0.76 at Diversified.

Features that differ most between false positives and true positives:
- cs_rank_vol_21: standardized gap -0.08
- intraday_range: standardized gap -0.07
- atr_pct_14: standardized gap -0.07
- vol_63: standardized gap -0.05
- dist_52w_low: standardized gap -0.05
- ret_126: standardized gap -0.05
- vol_21: standardized gap -0.04
- vix: standardized gap 0.04



## 7. Latest signals

In [9]:
latest = pipeline.step_latest()
latest.head(10)

Latest signals for 2026-11-20 (entry at next open)


 rank     symbol                         sector       price  pred_ens  above_threshold  regime_ok  stop_loss_price
    1 THANGAMAYL              Consumer Durables 3813.000000  0.367194             True          0      3507.959961
    2 SILVERBEES                        Unknown  146.779999  0.355813             True          0       135.039993
    3   SAREGAMA                        Unknown  666.700012  0.353118             True          0       613.359985
    4 SHIPROCKET                        Unknown  109.779999  0.350527             True          0       101.000000
    5    SANSERA Automobile and Auto Components 4798.399902  0.348032             True          0      4414.529785
    6 ATHERENERG Automobile and Auto Components 1935.400024  0.345883             True          0      1780.569946
    7      GLAND                     Healthcare 3331.899902  0.345046             True          0      3065.350098
    8  SBISILVER                        Unknown  149.160004  0.342706           

,rank,signal_date,symbol,sector,price,regime_ok,pred_dt,pred_rf,pred_xgb,pred_ens,above_threshold,threshold,stop_loss_price,hold_trading_days
0,1,2026-11-20,THANGAMAYL,Consumer Durables,3813.000000,0,0.370864,0.375073,0.359314,0.367194,True,0.300476,3507.959961,5
1,2,2026-11-20,SILVERBEES,Unknown,146.779999,0,0.370864,0.376401,0.335225,0.355813,True,0.300476,135.039993,5
2,3,2026-11-20,SAREGAMA,Unknown,666.700012,0,0.353609,0.363787,0.342449,0.353118,True,0.300476,613.359985,5
3,4,2026-11-20,SHIPROCKET,Unknown,109.779999,0,0.347030,0.361874,0.339181,0.350527,True,0.300476,101.000000,5
4,5,2026-11-20,SANSERA,Automobile and Auto Components,4798.399902,0,0.353609,0.345015,0.351050,0.348032,True,0.300476,4414.529785,5
5,6,2026-11-20,ATHERENERG,Automobile and Auto Components,1935.400024,0,0.353609,0.362657,0.329109,0.345883,True,0.300476,1780.569946,5
6,7,2026-11-20,GLAND,Healthcare,3331.899902,0,0.265108,0.345652,0.344440,0.345046,True,0.300476,3065.350098,5
7,8,2026-11-20,SBISILVER,Unknown,149.160004,0,0.370864,0.359869,0.325543,0.342706,True,0.300476,137.229996,5
8,9,2026-11-20,MAHABANK,Financial Services,99.449997,0,0.265108,0.346396,0.338760,0.342578,True,0.300476,91.489998,5
9,10,2026-11-20,HDFCSILVER,Unknown,145.350006,0,0.370864,0.361024,0.323072,0.342048,True,0.300476,133.720001,5
